# Humano × IA em português — versão revisada

Projeto acadêmico inspirado no artigo de Masih et al. A versão 2 compara seis configurações, calibra o modelo selecionado e pode retornar **Inconclusivo**. O domínio continua sendo redações de aprendizes de português, sem validação externa em escrita nativa.

Os resultados executados estão no pacote em `resultados_v2`. Este notebook contém o código completo, sem saídas de execução, e pode refazer o experimento em Python 3.12 / Google Colab. Não é necessário treinar novamente para usar o modelo já fornecido no VS Code.

## 1. Instalar dependências da etapa CPU

In [ ]:
%pip install -q numpy==2.5.4 pandas==3.1.0 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 joblib==1.6.0 nbformat==5.11.1

## 2. Gravar os módulos no ambiente do notebook

In [ ]:
%%writefile detector_pt.py
"""Experimento acadêmico humano x IA em português. Execute com --help."""
from __future__ import annotations

import argparse
import hashlib
import html
import itertools
import json
import re
import time
import unicodedata
from pathlib import Path
from urllib.request import urlopen

import joblib
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.svm import LinearSVC

REVISION = "69e5e82a5c0e2edd0b7765e04cb3a2c902962d8e"
DATA_URL = ("https://raw.githubusercontent.com/nxthang/"
    f"ProAID---Proficiency-Calibrated-AI-Detection/{REVISION}/"
    "PT-L2Detect%20Dataset/data/processed/")
HASHES = {
    "train": "6a1c9d00ddf9239df1b43dac76e52de478b327ba376b971f5eff92b7acb78eff",
    "val": "6541102035a594284d057afa03c06d7efd9f6e24148e70342fdeabd0b0ecd666",
    "test": "494e4642bbbad5d886675b2802188fc89bc68217c5f51361560adbb1eef37d97",
}
PROJECT_DIR = Path(__file__).resolve().parent


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def clean_text(text):
    """Preserva acentos, pontuação e caixa; retira marcação de transcrição."""
    if not isinstance(text, str):
        raise ValueError("Cada texto precisa ser uma string.")
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"</?(?:p|br|div)(?:\s[^>]*)?/?>", " ", text, flags=re.I)
    # Tags inline podem ocorrer DENTRO da palavra: ci<u>dade -> cidade.
    text = re.sub(r"</?(?:u|b|i|span|em|strong)(?:\s[^>]*)?/?>", "", text, flags=re.I)
    # Os corpora incluem trechos anotados entre <...>; preservamos o conteúdo.
    text = text.replace("<", "").replace(">", "")
    return re.sub(r"\s+", " ", text).strip()


def text_hash(text):
    return hashlib.sha256(clean_text(text).casefold().encode("utf-8")).hexdigest()


def load_data(data_dir):
    """Baixa uma revisão fixa, verifica bytes e elimina sobreposição entre splits."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    frames = {}
    for split, expected in HASHES.items():
        path = data_dir / f"{split}_full.jsonl"
        if not path.exists():
            with urlopen(DATA_URL + path.name, timeout=120) as response:
                payload = response.read()
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError(f"Arquivo remoto inesperado: {split}.")
            path.write_bytes(payload)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Checksum inválido: {path}. Use a revisão documentada.")
        records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        df = pd.DataFrame(records)
        if not df.is_ai.map(lambda x: isinstance(x, bool)).all():
            raise ValueError("Rótulo is_ai deve ser booleano.")
        if df.essay_id.isna().any():
            raise ValueError("Registro sem identificador.")
        df["label"] = df.is_ai.astype(int)
        df["text"] = df.text.map(clean_text)
        if df.text.str.len().eq(0).any():
            raise ValueError("Texto vazio após limpeza.")
        df["text_hash"] = df.text.map(text_hash)
        df["n_words"] = df.text.str.split().str.len()
        frames[split] = df
    combined = pd.concat(frames.values(), ignore_index=True)
    for key in ("essay_id", "text_hash"):
        if combined.groupby(key).label.nunique().gt(1).any():
            raise ValueError(f"Rótulos conflitantes para {key}; revise a base.")
    audit = {"dataset_revision": REVISION, "sha256": HASHES,
             "original_counts": {k: len(v) for k, v in frames.items()}, "overlaps_before": {}}
    for a, b in itertools.combinations(frames, 2):
        audit["overlaps_before"][f"{a}/{b}"] = {
            key: len(set(frames[a][key]) & set(frames[b][key]))
            for key in ("essay_id", "text_hash")}
    # Preserva primeiro o teste, depois validação; elimina cópias do treino.
    seen_ids, seen_hashes = set(), set()
    cleaned = {}
    for split in ("test", "val", "train"):
        keep = []
        for idx, row in frames[split].iterrows():
            if row.essay_id in seen_ids or row.text_hash in seen_hashes:
                continue
            seen_ids.add(row.essay_id)
            seen_hashes.add(row.text_hash)
            keep.append(idx)
        cleaned[split] = frames[split].loc[keep].reset_index(drop=True)
        if set(cleaned[split].label.unique()) != {0, 1}:
            raise ValueError(f"O split {split} precisa conter ambas as classes.")
    audit["clean_counts"] = {s: {"total": len(df), "human": int(df.label.eq(0).sum()),
        "ai": int(df.label.eq(1).sum())} for s, df in cleaned.items()}
    audit["removed"] = {s: len(frames[s])-len(cleaned[s]) for s in frames}
    audit["human_topic_missing"] = int(combined.loc[combined.label.eq(0), "topic_num"].isna().sum())
    for a, b in itertools.combinations(cleaned, 2):
        for key in ("essay_id", "text_hash"):
            if set(cleaned[a][key]) & set(cleaned[b][key]):
                raise AssertionError("Sobreposição após limpeza.")
    return cleaned, audit


def metrics(y, pred, score=None):
    y, pred = np.asarray(y), np.asarray(pred)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    result = {"n": len(y), "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) == 2 else None,
        "precision_ai": float(precision_score(y, pred, zero_division=0)),
        "recall_ai": float(recall_score(y, pred, zero_division=0)),
        "f1_ai": float(f1_score(y, pred, zero_division=0)),
        "f1_macro": float(f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),
        "false_positive_rate": float(fp/(tn+fp)) if tn+fp else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
    if score is not None and len(np.unique(y)) == 2:
        result.update(roc_auc=float(roc_auc_score(y, score)),
                      average_precision_ai=float(average_precision_score(y, score)))
    return result


def scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, list(model.classes_).index(1)]
    return model.decision_function(X)


def candidates(seed):
    return {
        "regressao_logistica": [LogisticRegression(C=c, max_iter=2000, class_weight="balanced", random_state=seed) for c in (0.5, 1., 2.)],
        "svm_linear": [LinearSVC(C=c, class_weight="balanced", random_state=seed, max_iter=5000) for c in (0.5, 1., 2.)],
        "random_forest": [RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
            max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=seed)],
        "dummy_majoritario": [DummyClassifier(strategy="most_frequent")],
    }


def paired_tests(predictions, y):
    comparisons = []
    for a, b in itertools.combinations(predictions, 2):
        ca, cb = predictions[a] == y, predictions[b] == y
        n_ab, n_ba = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
        p = float(binomtest(n_ab, n_ab+n_ba, .5).pvalue) if n_ab+n_ba else 1.
        comparisons.append({"model_a": a, "model_b": b, "a_only_correct": n_ab,
                            "b_only_correct": n_ba, "p_raw": p})
    previous = 0.
    for rank, item in enumerate(sorted(comparisons, key=lambda x: x["p_raw"])):
        previous = max(previous, min(1., item["p_raw"]*(len(comparisons)-rank)))
        item["p_holm"] = previous
        item["significant_0_05"] = previous < .05
    return comparisons


def bootstrap_ci(y, pred, n_boot=1000, seed=42):
    """IC condicionado a este conjunto; não mede mudança de domínio."""
    rng = np.random.default_rng(seed)
    y, pred = np.asarray(y), np.asarray(pred)
    groups = [np.flatnonzero(y == label) for label in (0, 1)]
    values = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        values.append([accuracy_score(y[idx], pred[idx]), f1_score(y[idx], pred[idx], average="macro")])
    ci = np.quantile(values, [.025, .975], axis=0)
    return {name: {"low": float(ci[0, i]), "high": float(ci[1, i])}
            for i, name in enumerate(("accuracy", "f1_macro"))}


def run_baselines(data_dir="data", output_dir="resultados", seeds=(7, 42, 123)):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    data, audit = load_data(data_dir)
    save_json(out / "auditoria_dados.json", audit)
    # Somente texto entra no classificador. Origem, modelo, nível e IDs ficam fora.
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=.98,
        max_features=30000, sublinear_tf=True, lowercase=True, strip_accents=None)
    matrices = {"train": vectorizer.fit_transform(data["train"].text)}
    matrices.update({s: vectorizer.transform(data[s].text) for s in ("val", "test")})
    selected, validation = {}, []
    for seed in seeds:
        for name, options in candidates(seed).items():
            best_score, best_model = -1., None
            started = time.perf_counter()
            for model in options:
                model.fit(matrices["train"], data["train"].label)
                val_score = f1_score(data["val"].label, model.predict(matrices["val"]), average="macro")
                if val_score > best_score:
                    best_score, best_model = val_score, model
            selected[(name, seed)] = best_model
            validation.append({"model": name, "seed": seed, "validation_f1_macro": best_score,
                "selected_C": best_model.get_params().get("C"), "fit_search_seconds": time.perf_counter()-started})
    validation = pd.DataFrame(validation)
    validation.to_csv(out / "selecao_validacao.csv", index=False)
    # A escolha da família usa apenas validação; o teste entra a partir daqui.
    winner = validation.groupby("model").validation_f1_macro.mean().idxmax()
    primary_seed = 42 if 42 in seeds else seeds[0]
    results, predictions, primary_preds = [], [], {}
    y_test = data["test"].label.to_numpy()
    for (name, seed), model in selected.items():
        started = time.perf_counter()
        pred = model.predict(matrices["test"])
        elapsed = time.perf_counter()-started
        score = scores(model, matrices["test"])
        results.append({"model": name, "seed": seed,
            "predict_batch_seconds": elapsed, **metrics(y_test, pred, score)})
        for row, prediction, value in zip(data["test"].itertuples(), pred, score):
            predictions.append({"essay_id": row.essay_id, "text_hash": row.text_hash,
                "model": name, "seed": seed, "true_label": row.label, "prediction": int(prediction),
                "score_ai": float(value), "score_type": "probability_uncalibrated" if hasattr(model,"predict_proba") else "svm_margin",
                "cefr_level": row.cefr_level, "n_words": row.n_words})
        if seed == primary_seed:
            primary_preds[name] = pred
        joblib.dump({"vectorizer": vectorizer, "classifier": model, "model_name": name,
            "seed": seed, "labels": {0: "Humano", 1: "IA"}, "data_revision": REVISION}, out / f"{name}_seed{seed}.joblib")
    result_df = pd.DataFrame(results)
    result_df.to_csv(out / "metricas_teste.csv", index=False)
    pd.DataFrame(predictions).to_csv(out / "predicoes_teste.csv", index=False)
    compare = {k: v for k, v in primary_preds.items() if k != "dummy_majoritario"}
    save_json(out / "mcnemar_holm.json", paired_tests(compare, y_test))
    summary = {"selected_by": "mean validation macro F1", "winner": winner,
        "primary_seed": primary_seed, "ci95_bootstrap": bootstrap_ci(y_test, primary_preds[winner]),
        "transformers_status": "not_trained", "warning": "Estudo exploratório de português L2; não comprova autoria."}
    save_json(out / "resumo.json", summary)
    joblib.dump(joblib.load(out / f"{winner}_seed{primary_seed}.joblib"), out / "modelo_selecionado.joblib")
    error_groups = []
    test = data["test"].copy()
    test["prediction"] = primary_preds[winner]
    test["length_bin"] = pd.cut(test.n_words, [0, 50, 100, 200, np.inf], labels=["1-50", "51-100", "101-200", "201+"])
    for column in ("cefr_level", "length_bin"):
        for group, frame in test.groupby(column, observed=True):
            error_groups.append({"group_type": column, "group": str(group), **metrics(frame.label, frame.prediction)})
    pd.DataFrame(error_groups).to_csv(out / "erros_por_grupo.csv", index=False)
    create_figures(result_df, primary_preds, y_test, out, primary_seed)
    print(json.dumps({"clean_counts": audit["clean_counts"], "winner": winner}, ensure_ascii=False, indent=2))
    print(result_df.loc[result_df.seed.eq(primary_seed), ["model", "accuracy", "f1_macro", "false_positive_rate"]].to_string(index=False))
    return data, result_df, summary


def create_figures(results, predictions, y, out, primary_seed):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    subset = results[results.seed.eq(primary_seed)].sort_values("f1_macro")
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.barh(subset.model.str.replace("_", " "), subset.f1_macro, color="#21736b")
    for i, value in enumerate(subset.f1_macro):
        ax.text(value+.01, i, f"{value:.3f}", va="center")
    ax.set(xlim=(0, 1.1), xlabel="F1 macro no teste", title=f"Humano × IA — português L2 (seed {primary_seed})")
    fig.tight_layout()
    fig.savefig(out / "comparacao_modelos.png", dpi=180)
    plt.close(fig)
    fig, axes = plt.subplots(1, len(predictions), figsize=(4*len(predictions), 4))
    for ax, (name, pred) in zip(np.atleast_1d(axes), predictions.items()):
        ConfusionMatrixDisplay.from_predictions(y, pred, labels=[0,1], display_labels=["Humano", "IA"], ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name.replace("_", " "), fontsize=10)
        ax.set_xlabel("Predito")
        ax.set_ylabel("Real")
    fig.tight_layout()
    fig.savefig(out / "matrizes_confusao.png", dpi=160)
    plt.close(fig)


def predict_text(model_path, text):
    text = clean_text(text)
    if not text:
        raise ValueError("Informe um texto não vazio.")
    bundle = joblib.load(model_path)  # Use apenas arquivos de modelo de origem confiável.
    if bundle.get("schema_version") == 2:
        from modelo_v2 import predict_bundle
        return predict_bundle(bundle, text)
    # Modelos antigos continuam legíveis, mas não devem decidir frases isoladas.
    if len(text.split()) < 50:
        return {"prediction": "Inconclusivo", "label": None, "score_ai": None,
                "reason": "Texto curto: esta versão exige pelo menos 50 palavras.",
                "model_version": "legado"}
    X = bundle["vectorizer"].transform([text])
    if X.nnz == 0:
        return {"prediction": "Inconclusivo", "reason": "Nenhum termo reconhecido no vocabulário."}
    model = bundle["classifier"]
    label = int(model.predict(X)[0])
    return {"prediction": bundle["labels"][label], "label": label,
        "score_ai": float(scores(model, X)[0]),
        "score_type": "probabilidade não calibrada" if hasattr(model,"predict_proba") else "margem SVM (não é probabilidade)",
        "scope": "Português L2; saída experimental, não comprovação de autoria."}


def explain_linear(model_path, text, top_n=12):
    bundle = joblib.load(model_path)
    model = bundle["classifier"]
    if not hasattr(model, "coef_"):
        raise ValueError("Use o modelo de regressão logística ou SVM linear.")
    vectorizer = bundle["vectorizer"]
    X = vectorizer.transform([clean_text(text)]).tocsr()
    contributions = X.data * model.coef_[0, X.indices]
    names = vectorizer.get_feature_names_out()[X.indices]
    order = np.argsort(np.abs(contributions))[::-1][:top_n]
    return pd.DataFrame({"termo": names[order], "contribuicao_para_IA": contributions[order]})


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    train = sub.add_parser("train", help="Treinar a versão revisada com calibração e abstenção")
    train.add_argument("--data-dir", default=str(PROJECT_DIR / "data"))
    train.add_argument("--output-dir", default=str(PROJECT_DIR / "resultados_v2"))
    baseline = sub.add_parser("train-baselines", help="Experimento clássico de referência")
    baseline.add_argument("--data-dir", default=str(PROJECT_DIR / "data"))
    baseline.add_argument("--output-dir", default=str(PROJECT_DIR / "resultados_baselines_revisados"))
    pred = sub.add_parser("predict", help="Classificar um texto com o modelo já treinado")
    pred.add_argument("--model", default=str(PROJECT_DIR / "resultados_v2/modelo_selecionado.joblib"))
    source = pred.add_mutually_exclusive_group(required=True)
    source.add_argument("--text")
    source.add_argument("--text-file", help="Arquivo .txt em UTF-8")
    evaluate = sub.add_parser("evaluate", help="Avaliar uma base externa rotulada, sem retreinar")
    evaluate.add_argument("--csv", required=True, help="CSV com colunas text,label (0 humano; 1 IA)")
    evaluate.add_argument("--model", default=str(PROJECT_DIR / "resultados_v2/modelo_selecionado.joblib"))
    evaluate.add_argument("--output-dir", default=str(PROJECT_DIR / "avaliacao_externa"))
    args = parser.parse_args()
    try:
        if args.command == "train":
            from modelo_v2 import run_improved
            run_improved(args.data_dir, args.output_dir)
        elif args.command == "train-baselines":
            run_baselines(args.data_dir, args.output_dir)
        elif args.command == "evaluate":
            from modelo_v2 import evaluate_external
            evaluate_external(args.model, args.csv, args.output_dir)
        else:
            text = Path(args.text_file).read_text(encoding="utf-8-sig") if args.text_file else args.text
            print(json.dumps(predict_text(args.model, text), ensure_ascii=False, indent=2))
    except (ValueError, FileNotFoundError) as exc:
        parser.exit(2, f"Erro: {exc}\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile modelo_v2.py
"""Seleção em treino, calibração independente e previsão com abstenção."""
from pathlib import Path
import json
import math
import re
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.frozen import FrozenEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

from detector_pt import clean_text, load_data, metrics, save_json, text_hash

SCOPE = "Experimental: redações de português L2. Não validado para autoria geral em português nativo."


def prune_near_duplicates(frames, threshold=.95):
    """Filtro lexical conservador; não identifica paráfrases ou autores."""
    joined = pd.concat([frames[s].assign(split=s) for s in ("test", "val", "train")], ignore_index=True)
    # Esta representação serve somente à auditoria; não entra no classificador.
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                        max_features=30000).fit_transform(joined.text)
    parent = list(range(len(joined)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    pairs = []
    for offset in range(0, len(joined), 128):
        similarity = (X[offset:offset+128] @ X.T).tocoo()
        keep = (similarity.data >= threshold) & (similarity.row+offset < similarity.col)
        for r, c, value in zip(similarity.row[keep]+offset, similarity.col[keep], similarity.data[keep]):
            a, b = find(int(r)), find(int(c))
            parent[max(a,b)] = min(a,b)
            pairs.append({"id_a": joined.iloc[r].essay_id, "id_b": joined.iloc[c].essay_id,
                "split_a": joined.iloc[r].split, "split_b": joined.iloc[c].split,
                "similarity": float(value), "same_label": bool(joined.iloc[r].label == joined.iloc[c].label)})
    components = {}
    for i in range(len(joined)):
        components.setdefault(find(i), []).append(i)
    retained, conflicts = [], 0
    for members in components.values():
        if joined.iloc[members].label.nunique() > 1:
            conflicts += len(members)  # Ambiguidade: excluir todo o grupo.
        else:
            retained.append(min(members))
    result = joined.iloc[sorted(retained)]
    cleaned = {s: result[result.split.eq(s)].drop(columns="split").reset_index(drop=True) for s in frames}
    report = {"cosine_threshold": threshold, "near_pairs": pairs,
        "removed_by_split": {s: len(frames[s])-len(cleaned[s]) for s in frames},
        "conflicting_rows_removed": conflicts,
        "note": "Filtro lexical de auditoria, ajustado sem rótulos; não garante ausência de similaridade semântica."}
    return cleaned, report


def feature_candidates():
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000,
        sublinear_tf=True, lowercase=True, strip_accents=None)
    chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
        max_features=50000, sublinear_tf=True)
    hybrid = FeatureUnion([("words", clone(words)), ("chars", clone(chars))],
        transformer_weights={"words": 1., "chars": 1.})
    lr = LogisticRegression(C=1., max_iter=2000, class_weight="balanced", random_state=42)
    svm = LinearSVC(C=1., class_weight="balanced", max_iter=5000, random_state=42)
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, max_features="sqrt",
                                class_weight="balanced", random_state=42, n_jobs=-1)
    return {name: Pipeline([("features", clone(features)), ("classifier", clone(model))])
        for name, features, model in [
            ("palavras_logistica", words, lr), ("palavras_svm", words, svm),
            ("palavras_random_forest", words, rf), ("caracteres_logistica", chars, lr),
            ("hibrido_logistica", hybrid, lr), ("hibrido_svm", hybrid, svm)]}


def word_tokens(text):
    return re.findall(r"(?u)\b\w\w+\b", text.casefold())


def vocabulary_coverage(text, vocabulary):
    tokens = set(word_tokens(text))
    return sum(t in vocabulary for t in tokens)/len(tokens) if tokens else 0.


def support_check(bundle, text):
    policy = bundle["policy"]
    count = len(text.split())
    tokens = word_tokens(text)
    coverage = vocabulary_coverage(text, bundle["vocabulary"])
    reasons = []
    if count < policy["min_words"]:
        reasons.append(f"Texto curto: mínimo operacional de {policy['min_words']} palavras; recebido {count}.")
    if count > policy["max_words"]:
        reasons.append(f"Texto além do comprimento estudado: máximo {policy['max_words']} palavras; recebido {count}.")
    if coverage < policy["min_vocabulary_coverage"]:
        reasons.append("Pouca cobertura do vocabulário de treinamento.")
    if not tokens or len(set(tokens))/len(tokens) < .15:
        reasons.append("Texto vazio de palavras ou excessivamente repetitivo.")
    return {"n_words": count, "vocabulary_coverage": float(coverage), "reasons": reasons}


def conformal_pvalues(reference, probabilities):
    """p-valores de compatibilidade por classe; NÃO são probabilidades de autoria."""
    values = []
    for label in (0, 1):
        scores = np.asarray(reference[str(label)], dtype=float)
        if not len(scores):
            raise ValueError("Falta referência de uma das classes para abstenção.")
        candidate_score = 1-float(probabilities[label])
        values.append(float((1+np.sum(scores >= candidate_score))/(len(scores)+1)))
    return values


def decide(bundle, text, probabilities=None):
    support = support_check(bundle, text)
    result = {"prediction": "Inconclusivo", "label": None, "score_ai": None,
        "score_type": "não calculado fora das condições operacionais",
        "model_version": 2, "scope": SCOPE, **support}
    if support["reasons"]:
        return result
    if probabilities is None:
        probabilities = bundle["calibrated_model"].predict_proba([text])[0]
    pvalues = conformal_pvalues(bundle["conformal_reference"], probabilities)
    compatible = [k for k, p in enumerate(pvalues) if p > bundle["policy"]["alpha"]]
    result.update(score_ai=float(probabilities[1]), score_type="probabilidade calibrada no domínio L2; não comprova autoria",
        compatible_labels=[bundle["labels"][k] for k in compatible],
        compatibility_pvalues={"Humano": pvalues[0], "IA": pvalues[1]})
    if len(compatible) == 1:
        result.update(prediction=bundle["labels"][compatible[0]], label=compatible[0])
    else:
        result["reasons"] = ["Ambas as classes são compatíveis com a referência." if compatible
                             else "Nenhuma classe tem compatibilidade suficiente com a referência."]
    return result


def predict_bundle(bundle, text):
    if bundle.get("schema_version") != 2:
        raise ValueError("Use um modelo da versão 2.")
    if not isinstance(text, str) or len(text) > 100000:
        raise ValueError("Informe uma string de até 100.000 caracteres.")
    text = clean_text(text)
    if not text:
        raise ValueError("Informe um texto não vazio.")
    return decide(bundle, text)


def explain_improved(model_path, text, top_n=15):
    """Contribuições exatas para a margem linear antes da calibração."""
    bundle = joblib.load(model_path)
    if bundle.get("schema_version") != 2:
        raise ValueError("Use um modelo v2.")
    base = bundle["base_model"]
    classifier = base.named_steps["classifier"]
    if not hasattr(classifier,"coef_"):
        raise ValueError("O modelo selecionado não é linear; use uma explicação apropriada a árvores.")
    features = base.named_steps["features"]
    X = features.transform([clean_text(text)]).tocsr()
    contributions = X.data * classifier.coef_[0,X.indices]
    names = features.get_feature_names_out()[X.indices]
    order = np.argsort(np.abs(contributions))[::-1][:top_n]
    return {"note": "Margem antes da calibração; contribuição não é evidência de autoria.",
        "intercept": float(classifier.intercept_[0]), "all_features_sum": float(contributions.sum()),
        "raw_margin": float(classifier.decision_function(X)[0]),
        "top_features": [{"feature": str(names[i]), "contribution": float(contributions[i])} for i in order]}


def selection_metrics(labels, decisions):
    y = np.asarray(labels, dtype=int)
    predicted = np.array([-1 if d["label"] is None else d["label"] for d in decisions])
    accepted = predicted >= 0
    return {"n_total": len(y), "n_decided": int(accepted.sum()), "n_inconclusive": int((~accepted).sum()),
        "decision_coverage": float(accepted.mean()),
        "human_decision_coverage": float(accepted[y==0].mean()) if np.any(y==0) else None,
        "ai_decision_coverage": float(accepted[y==1].mean()) if np.any(y==1) else None,
        "false_ai_among_all_humans": float(np.mean(predicted[y==0] == 1)) if np.any(y==0) else None,
        "false_human_among_all_ai": float(np.mean(predicted[y==1] == 0)) if np.any(y==1) else None,
        "conditional_metrics_decided_only": metrics(y[accepted], predicted[accepted]) if accepted.any() else None}


def probability_metrics(y, p):
    return {"brier": float(brier_score_loss(y, p)),
            "log_loss": float(log_loss(y, np.column_stack([1-p,p]), labels=[0,1]))}


def run_improved(data_dir="data", output_dir="resultados_v2"):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    frames, exact_audit = load_data(data_dir)
    known_hashes = set().union(*(set(f.text_hash) for f in frames.values()))
    frames, near_audit = prune_near_duplicates(frames)
    save_json(out/"auditoria_dados.json", {"exact": exact_audit, "near": near_audit,
        "effective_counts": {s: {"total": len(f), "human": int(f.label.eq(0).sum()), "ai": int(f.label.eq(1).sum())} for s,f in frames.items()}})
    train = frames["train"]
    # Seleção de família SOMENTE por CV do treino, com vetorizador dentro do pipeline.
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    candidate_models = feature_candidates()
    rows = []
    for name, model in candidate_models.items():
        start = time.perf_counter()
        res = cross_validate(model, train.text.tolist(), train.label.to_numpy(), cv=cv,
            scoring={"f1_macro": "f1_macro", "balanced_accuracy": "balanced_accuracy"},
            error_score="raise", n_jobs=1)
        rows.append({"model": name, "cv_f1_macro": float(res["test_f1_macro"].mean()),
            "cv_f1_sd": float(res["test_f1_macro"].std(ddof=1)),
            "cv_balanced_accuracy": float(res["test_balanced_accuracy"].mean()),
            "seconds": time.perf_counter()-start})
        print(f"CV {name}: {rows[-1]['cv_f1_macro']:.4f}", flush=True)
    pd.DataFrame(rows).to_csv(out/"comparacao_cv_treino.csv", index=False)
    winner = max(rows, key=lambda row: row["cv_f1_macro"])["model"]
    base = clone(candidate_models[winner]).fit(train.text.tolist(), train.label.to_numpy())
    probability_cal, uncertainty_cal = train_test_split(frames["val"], test_size=.5,
        random_state=42, stratify=frames["val"].label)
    calibrated = CalibratedClassifierCV(FrozenEstimator(base), method="sigmoid")
    calibrated.fit(probability_cal.text.tolist(), probability_cal.label.to_numpy())
    probs_reference = calibrated.predict_proba(uncertainty_cal.text.tolist())
    y_reference = uncertainty_cal.label.to_numpy()
    reference = {str(k): (1-probs_reference[y_reference==k,k]).tolist() for k in (0,1)}
    vocabulary = set(TfidfVectorizer(min_df=2).fit(train.text.tolist()).vocabulary_)
    train_coverage = [vocabulary_coverage(t,vocabulary) for t in train.text]
    bundle = {"schema_version": 2, "base_model": base, "calibrated_model": calibrated,
        "model_name": winner, "labels": {0: "Humano", 1: "IA"},
        "policy": {"min_words": 50, "max_words": int(train.n_words.max()),
            "min_vocabulary_coverage": float(max(.5,min(.9,np.quantile(train_coverage,.01)-.05))), "alpha": .05},
        "vocabulary": vocabulary, "conformal_reference": reference, "known_data_hashes": known_hashes,
        "scope": SCOPE}
    joblib.dump(bundle, out/"modelo_selecionado.joblib", compress=3)
    for name, frame in {"train": train, "calibration_probability": probability_cal,
                         "calibration_uncertainty": uncertainty_cal, "test": frames["test"]}.items():
        frame[["essay_id","text_hash","label"]].to_csv(out/f"ids_{name}.csv",index=False)
    # O teste histórico já foi visto na v1: esta avaliação é diagnóstica, não um novo holdout.
    test = frames["test"]
    p = calibrated.predict_proba(test.text.tolist())
    decisions = [decide(bundle, t, prob) for t,prob in zip(test.text,p)]
    report = {"winner_selected_on_training_cv": winner, "selection": "3-fold stratified CV macro F1",
        "test_status": "Diagnóstico no teste histórico; validação externa independente ainda necessária.",
        "partitions": {"train": len(train), "probability_calibration": len(probability_cal),
                       "uncertainty_calibration": len(uncertainty_cal), "test": len(test)},
        "binary_default_0_5_all_test": metrics(test.label, p[:,1]>=.5, p[:,1]),
        "calibrated_probability_metrics": probability_metrics(test.label,p[:,1]),
        "selective": selection_metrics(test.label, decisions), "policy": bundle["policy"],
        "reference_per_class": {k:len(v) for k,v in reference.items()},
        "conformal_note": "Compatibilidade condicionada à classe requer dados intercambiáveis com a referência. Sem garantia para português nativo ou textos fora do domínio; abstenção não é acerto.",
        "transformer_status": "not_trained"}
    if hasattr(base,"predict_proba"):
        report["uncalibrated_probability_metrics"] = probability_metrics(test.label,base.predict_proba(test.text.tolist())[:,1])
    save_json(out/"relatorio_v2.json", report)
    pd.DataFrame([{"essay_id": row.essay_id, "text_hash": row.text_hash, "true_label": row.label,
        "n_words": row.n_words, "probability_ai": float(prob[1]), "prediction": decision["prediction"],
        "predicted_label": decision["label"], "reasons": " | ".join(decision["reasons"])}
        for row,prob,decision in zip(test.itertuples(),p,decisions)]).to_csv(out/"predicoes_teste.csv",index=False)
    print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)
    return report


def evaluate_external(model_path, csv_path, output_dir):
    bundle = joblib.load(model_path)
    if bundle.get("schema_version") != 2:
        raise ValueError("A avaliação externa requer um modelo v2.")
    df = pd.read_csv(csv_path)
    if not {"text","label"}.issubset(df.columns) or df.empty:
        raise ValueError("CSV deve conter linhas e colunas text,label.")
    if df.text.isna().any() or not df.label.isin([0,1]).all():
        raise ValueError("Textos obrigatórios e rótulos 0/1 são necessários.")
    df["text"] = df.text.map(clean_text)
    if df.text.str.len().eq(0).any():
        raise ValueError("Há texto vazio no CSV.")
    df["hash"] = df.text.map(text_hash)
    if df.hash.duplicated().any():
        raise ValueError("Há textos repetidos na base externa; revise antes de avaliar.")
    if set(df.hash) & bundle["known_data_hashes"]:
        raise ValueError("A base externa contém textos já presentes no experimento original.")
    decisions = [predict_bundle(bundle, t) for t in df.text]
    report = selection_metrics(df.label, decisions)
    report["scope"] = "Base fornecida; conferir origem, grupos, temas, autores e geradores manualmente. Não foi usada para ajuste."
    out = Path(output_dir)
    out.mkdir(parents=True,exist_ok=True)
    save_json(out/"relatorio_externo.json",report)
    pd.DataFrame([{"row": i, "true_label": int(y), "text_hash": h, **d}
        for i,(y,h,d) in enumerate(zip(df.label,df.hash,decisions))]).to_json(out/"predicoes_externas.json",orient="records",force_ascii=False,indent=2)
    print(json.dumps(report,ensure_ascii=False,indent=2))
    return report


In [ ]:
%%writefile transformer_pt.py
"""Extensão opcional: fine-tuning, calibração e limiar, sem usar teste para ajuste."""
import argparse
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.metrics import brier_score_loss, f1_score, log_loss, precision_recall_curve
from sklearn.model_selection import train_test_split

from detector_pt import load_data, metrics, save_json, clean_text

MODELS = {
    "bertimbau": ("neuralmind/bert-base-portuguese-cased", "94d69c95f98f7d5b2a8700c420230ae10def0baa"),
    "xlm_roberta": ("FacebookAI/xlm-roberta-base", "e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"),
}


def ece_binary(y, p, bins=10):
    """ECE da probabilidade da classe IA, em intervalos fixos, incluindo 0 e 1."""
    y, p = np.asarray(y), np.asarray(p)
    if len(y) == 0 or y.shape != p.shape or bins < 1 or not np.isin(y, [0,1]).all():
        raise ValueError("Rótulos/probabilidades inválidos para ECE.")
    if not np.isfinite(p).all() or np.any((p<0)|(p>1)):
        raise ValueError("Probabilidades devem estar entre 0 e 1.")
    assignments = np.minimum((p * bins).astype(int), bins-1)
    return float(sum(np.mean(assignments == k) * abs(p[assignments == k].mean()-y[assignments == k].mean())
                     for k in range(bins) if np.any(assignments == k)))


def calibration_metrics(y, p):
    return {"brier": float(brier_score_loss(y, p)),
            "log_loss": float(log_loss(y, np.column_stack([1-p, p]), labels=[0, 1])),
            "ece_binary_10_bins": ece_binary(y, p)}


def fit_temperature(logits, labels):
    logits, labels = np.asarray(logits), np.asarray(labels)
    if logits.shape != (len(labels), 2) or set(labels) != {0,1} or not np.isfinite(logits).all():
        raise ValueError("Calibração exige logits finitos (n,2) e ambas as classes.")
    result = minimize_scalar(lambda log_t: log_loss(labels, softmax(logits / np.exp(log_t), axis=1), labels=[0, 1]),
                             bounds=(math.log(.05), math.log(20)), method="bounded")
    if not result.success:
        raise RuntimeError("A calibração de temperatura não convergiu.")
    return float(np.exp(result.x))


def choose_threshold(y, p, target_precision=.90, minimum_positives=10):
    y, p = np.asarray(y), np.asarray(p)
    if y.shape != p.shape or not len(y) or not 0 < target_precision <= 1 or minimum_positives < 1:
        raise ValueError("Parâmetros inválidos para ajuste de limiar.")
    if not np.isin(y,[0,1]).all() or not np.isfinite(p).all() or np.any((p<0)|(p>1)):
        raise ValueError("Rótulos/probabilidades inválidos.")
    if len(np.unique(y)) < 2:
        return {"threshold": None, "status": "Não é possível validar o limiar sem ambas as classes."}
    precision, recall, thresholds = precision_recall_curve(y, p)
    valid = [i for i, t in enumerate(thresholds)
             if precision[i] >= target_precision and np.sum(p >= t) >= minimum_positives]
    if not valid:
        return {"threshold": None, "status": "Meta não atingida no subconjunto de ajuste."}
    i = max(valid, key=lambda i: (recall[i], precision[i], thresholds[i]))
    return {"threshold": float(thresholds[i]), "tuning_precision": float(precision[i]),
            "tuning_recall": float(recall[i]), "status": "Meta observada no ajuste; não é garantia no teste."}


def train_transformer(name="bertimbau", data_dir="data", output_dir="resultados_transformer", seed=42, epochs=3, allow_cpu=False):
    if name not in MODELS or epochs < 1:
        raise ValueError("Modelo ou número de épocas inválido.")
    import torch
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
        DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
    if not torch.cuda.is_available() and not allow_cpu:
        raise RuntimeError("Ative uma GPU no Colab/Kaggle. CPU pode ser usada com allow_cpu=True, mas será lenta.")
    set_seed(seed)
    out = Path(output_dir) / f"{name}_seed{seed}"
    out.mkdir(parents=True, exist_ok=True)
    data, audit = load_data(data_dir)
    from modelo_v2 import prune_near_duplicates
    data, near_audit = prune_near_duplicates(data)
    audit["near_duplicate_audit"] = near_audit
    # Três subconjuntos distintos: checkpoint, temperatura e limiar.
    checkpoint_val, remainder = train_test_split(data["val"], test_size=.5, random_state=seed, stratify=data["val"].label)
    calibration, threshold_data = train_test_split(remainder, test_size=.5, random_state=seed, stratify=remainder.label)
    partitions = {"train": data["train"], "checkpoint": checkpoint_val,
                  "calibration": calibration, "threshold": threshold_data, "test": data["test"]}
    model_id, revision = MODELS[name]
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    truncation_audit = {}
    for key, frame in partitions.items():
        lengths = [len(ids) for ids in tokenizer(frame.text.tolist(), truncation=False)["input_ids"]]
        truncation_audit[key] = {"n": len(lengths), "over_256_tokens": int(np.sum(np.asarray(lengths)>256))}
    model = AutoModelForSequenceClassification.from_pretrained(model_id, revision=revision,
        num_labels=2, id2label={0: "Humano", 1: "IA"}, label2id={"Humano": 0, "IA": 1})
    def encode(frame):
        dataset = Dataset.from_dict({"text": frame.text.tolist(), "labels": frame.label.tolist()})
        return dataset.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=256), batched=True, remove_columns=["text"])
    encoded = {k: encode(v) for k, v in partitions.items()}
    args = TrainingArguments(output_dir=str(out / "checkpoints"), learning_rate=3e-5,
        per_device_train_batch_size=8, gradient_accumulation_steps=4,
        per_device_eval_batch_size=16, num_train_epochs=epochs,
        weight_decay=.01, warmup_ratio=.1, max_grad_norm=1., optim="adamw_torch",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        fp16=torch.cuda.is_available(), seed=seed, data_seed=seed, report_to="none",
        dataloader_num_workers=0)
    trainer = Trainer(model=model, args=args, train_dataset=encoded["train"], eval_dataset=encoded["checkpoint"],
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=lambda p: {"f1_macro": f1_score(p.label_ids, np.argmax(p.predictions, axis=1), average="macro")})
    trainer.train()
    logits_cal = trainer.predict(encoded["calibration"]).predictions
    temperature = fit_temperature(logits_cal, calibration.label.to_numpy())
    logits_threshold = trainer.predict(encoded["threshold"]).predictions
    threshold_info = choose_threshold(threshold_data.label.to_numpy(), softmax(logits_threshold/temperature, axis=1)[:,1])
    # Teste consultado só após congelar checkpoint, temperatura e limiar.
    logits_test = trainer.predict(encoded["test"]).predictions
    raw = softmax(logits_test, axis=1)[:, 1]
    calibrated = softmax(logits_test/temperature, axis=1)[:, 1]
    y = data["test"].label.to_numpy()
    report = {"model": name, "model_id": model_id, "revision": revision, "seed": seed,
        "epochs": epochs, "max_length": 256, "effective_batch_size_single_gpu": 32,
        "temperature": temperature, "threshold_tuning": threshold_info,
        "partition_sizes": {k: len(v) for k, v in partitions.items()},
        "truncation_audit": truncation_audit,
        "test_status": "Teste histórico já examinado; requer validação externa independente.",
        "default_threshold_0_5": metrics(y, calibrated >= .5, calibrated),
        "calibration_before": calibration_metrics(y, raw),
        "calibration_after": calibration_metrics(y, calibrated)}
    threshold = threshold_info["threshold"]
    if threshold is not None:
        report["tuned_threshold_metrics"] = metrics(y, calibrated >= threshold, calibrated)
    trainer.save_model(str(out / "model"))
    tokenizer.save_pretrained(out / "model")
    save_json(out / "relatorio.json", report)
    save_json(out / "auditoria_dados.json", audit)
    for key, frame in partitions.items():
        frame[["essay_id", "text_hash", "label"]].to_csv(out / f"ids_{key}.csv", index=False)
    pd.DataFrame({"essay_id": data["test"].essay_id, "text_hash": data["test"].text_hash,
        "true_label": y, "probability_raw": raw, "probability_calibrated": calibrated,
        "prediction_default": (calibrated >= .5).astype(int)}).to_csv(out / "predicoes_teste.csv", index=False)
    return report


def load_predictor(model_dir, temperature=None):
    # Sem temperatura explícita, usar a do treinamento; nunca voltar silenciosamente a 1.
    if temperature is None:
        report_path = Path(model_dir).parent / "relatorio.json"
        if not report_path.exists():
            raise ValueError("Relatório de calibração ausente; informe temperatura explicitamente.")
        temperature = json.loads(report_path.read_text(encoding="utf-8"))["temperature"]
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperatura precisa ser positiva e finita.")
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    def predict_proba(texts):
        if isinstance(texts, str):
            raise ValueError("Passe uma lista de textos, não uma string isolada.")
        output = []
        for start in range(0, len(texts), 16):
            batch = [clean_text(str(t)) for t in texts[start:start+16]]
            inputs = tokenizer(batch, truncation=True, max_length=256, padding=True, return_tensors="pt").to(device)
            with torch.no_grad():
                output.append(torch.softmax(model(**inputs).logits/temperature, dim=-1).cpu().numpy())
        return np.concatenate(output) if output else np.empty((0,2))
    return tokenizer, predict_proba


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", choices=MODELS, default="bertimbau")
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output-dir", default="resultados_transformer")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--allow-cpu", action="store_true")
    a = parser.parse_args()
    report = train_transformer(a.model, a.data_dir, a.output_dir, a.seed, a.epochs, a.allow_cpu)
    print(report)


## 3. Treinar a versão revisada

O código baixa uma revisão fixa e valida os arquivos. Remove duplicatas exatas e quase duplicatas lexicais. Seleciona entre palavras, caracteres e combinação de ambos com regressão logística, SVM e Random Forest, usando validação cruzada apenas no treino. Cada fold ajusta seu próprio vocabulário.

A validação publicada é dividida em dois grupos disjuntos: calibração sigmoide e referência de compatibilidade. Há 50 palavras mínimas operacionais, porque não existem textos de IA mais curtos na base. As restrições de comprimento, vocabulário e repetição são controles de suporte, não detectores de autoria.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from modelo_v2 import run_improved, explain_improved
from detector_pt import predict_text

DATA_DIR=Path("data")
RESULTS_DIR=Path("resultados_v2")
report=run_improved(DATA_DIR,RESULTS_DIR)
display(pd.read_csv(RESULTS_DIR/"comparacao_cv_treino.csv"))

## 4. Avaliar acertos E cobertura

Há duas avaliações distintas: a classificação forçada em 0,5, para diagnóstico; e a saída operacional com abstenção. A acurácia dos casos decididos não pode ser anunciada como acurácia de todos os textos. Uma abstenção não conta como acerto.

O teste já foi visto na versão anterior: ele é um diagnóstico histórico. A superioridade generalizável requer uma base externa nova. Os p-valores de compatibilidade não são probabilidades de autoria; sua interpretação depende de exemplos comparáveis aos da referência.

In [ ]:
display(report["binary_default_0_5_all_test"])
display(report["selective"])
display(report["calibrated_probability_metrics"])
display(report["policy"])
display(json.loads((RESULTS_DIR/"auditoria_dados.json").read_text(encoding="utf-8"))["effective_counts"])

## 5. Analisar seu texto

Cole um parágrafo ou redação. As frases curtas abaixo de 50 palavras retornam Inconclusivo, qualquer que seja sua origem. O modelo deste pacote foi estudado até 535 palavras. Fora desses limites ele não fornece pontuação. Uma saída dentro desses limites também não comprova autoria.

In [ ]:
texto="Hoje fui ao mercado comprar pão e frutas para o café da manhã."
display(predict_text(RESULTS_DIR/"modelo_selecionado.joblib",texto))

## 6. Explicar a margem do modelo linear

`words__` identifica palavras/bigramas; `chars__` identifica trechos de palavras. As contribuições, somadas ao intercepto, reconstroem a margem antes da calibração. Não são provas de autoria. Esta explicação é apenas diagnóstica, inclusive quando a saída operacional é Inconclusivo.

In [ ]:
explanation=explain_improved(RESULTS_DIR/"modelo_selecionado.joblib",texto)
display(pd.DataFrame(explanation["top_features"]))
print("Intercepto:",explanation["intercept"],"Margem:",explanation["raw_margin"])

## 7. Avaliação externa

Para testar generalização, use um CSV novo com `text,label` (0 humano, 1 IA). Os rótulos precisam vir de origem conhecida, nunca de outro detector. Guarde autores/fontes, tema, gerador e família de prompt; mantenha grupos relacionados juntos. A base externa não deve ser usada para ajustar o modelo e continuar sendo chamada de teste.

O comando abaixo rejeita duplicatas exatas e textos já presentes no experimento. Ainda é necessário auditar paráfrases, autores e assuntos.

In [ ]:
AVALIAR_BASE_EXTERNA=False
if AVALIAR_BASE_EXTERNA:
    from modelo_v2 import evaluate_external
    evaluate_external(RESULTS_DIR/"modelo_selecionado.joblib","base_externa.csv","avaliacao_externa")

## 8. Transformers opcionais — não treinados nesta entrega

Ative uma GPU antes de habilitar esta etapa. O código foi revisado para usar a mesma limpeza e deduplicação, separar seleção/calibração/limiar e registrar quantos textos ultrapassam 256 tokens. A inferência da extensão usa os primeiros 256 tokens; a contagem de truncamento precisa ser discutida no trabalho.

Trocar o modelo não corrige sozinho a diferença entre aprendizes e falantes nativos. A extensão tem avaliação binária própria e ainda não possui a mesma política operacional de abstenção da versão CPU.

In [ ]:
EXECUTAR_TRANSFORMERS=False
if EXECUTAR_TRANSFORMERS:
    import subprocess,sys
    subprocess.check_call([sys.executable,"-m","pip","install","torch>=2.6,<3","transformers==4.57.1",
        "datasets>=3,<5","accelerate>=1,<2","sentencepiece>=0.2,<1"])
    from transformer_pt import train_transformer
    for model in ["bertimbau","xlm_roberta"]:
        for seed in [7,42,123]:
            result=train_transformer(model,DATA_DIR,"resultados_transformer",seed=seed)
            display(result)

## 9. Explicabilidade local opcional dos transformers

Após treinar, habilite para inspecionar uma previsão. A temperatura será carregada automaticamente do relatório, e a amostra é limitada ao mesmo orçamento de 256 tokens. LIME/SHAP não foram executados nesta revisão.

In [ ]:
EXECUTAR_XAI=False
if EXECUTAR_XAI:
    import subprocess,sys
    subprocess.check_call([sys.executable,"-m","pip","install","lime==0.2.0.1","shap>=0.46,<1"])
    import shap
    from lime.lime_text import LimeTextExplainer
    from transformer_pt import load_predictor
    model_dir=Path("resultados_transformer/bertimbau_seed42/model")
    tokenizer,predict_proba=load_predictor(model_dir)
    sample=tokenizer.decode(tokenizer.encode(texto,truncation=True,max_length=256),skip_special_tokens=True)
    lime=LimeTextExplainer(class_names=["Humano","IA"],random_state=42)
    explanation=lime.explain_instance(sample,predict_proba,labels=(1,),num_features=12,num_samples=300)
    display(pd.DataFrame(explanation.as_list(label=1),columns=["termo","peso"]))
    explainer=shap.Explainer(predict_proba,shap.maskers.Text(tokenizer),output_names=["Humano","IA"])
    values=explainer([sample],max_evals=128)
    shap.plots.text(values[0,:,1])

## Fontes e próximos experimentos

- [Artigo-base](https://pmc.ncbi.nlm.nih.gov/articles/PMC12686477/).
- [PT-L2Detect](https://github.com/nxthang/ProAID---Proficiency-Calibrated-AI-Detection/tree/69e5e82a5c0e2edd0b7765e04cb3a2c902962d8e/PT-L2Detect%20Dataset).
- [Calibração no scikit-learn](https://scikit-learn.org/stable/modules/calibration.html).
- [Introdução à predição conformal](https://arxiv.org/abs/2107.07511).
- [BERTimbau](https://github.com/neuralmind-ai/portuguese-bert).

Prioridade para o próximo estudo: textos nativos em português com origem documentada, temas e comprimentos comparáveis, separação por autor/prompt, teste externo e geradores não vistos. As fontes humanas desta base continuam confundidas com o rótulo; tópicos humanos e IDs de autores não puderam ser verificados. Resultados desta base não validam atribuição de autoria no mundo real.